# Tool-Router Ladder: Stage F4, GPTQ 4-bit checkpoints

`src/quantize.py`: merge each size's SFT adapter into its base and quantize it to **GPTQ W4A16**
(llm-compressor, compressed-tensors format, group size 128), calibrated on 128 **train-split** states
(cut to their last 2,048 tokens). Each checkpoint goes to the Hub as
`<hf-user>/tool-router-<size>-sft-gptq`, which `kaggle_serve_bench.ipynb` then serves with vLLM.

**Settings:** Accelerator **GPU T4** (one is enough), Internet **on**, the **`tool-router-grpo-states`**
dataset attached (the train states, the same one `kaggle_grpo` uses), secret `HF_TOKEN` (write) ticked.
**Run:** Save Version -> **Save & Run All (Commit)**. **No API quota.** Estimate ~1-1.5 GPU h for all three
sizes. A size whose Hub repo already holds `quantize.json` is skipped, so a rerun resumes.

**Send back:** the smoke cell's output if it fails; otherwise nothing (the `quantize.json` lines print at the end).

In [ ]:
# Secrets: Add-ons -> Secrets, and TICK each one for this notebook.
import os
from kaggle_secrets import UserSecretsClient
_s = UserSecretsClient()
for key in ["HF_TOKEN"]:
    os.environ[key] = _s.get_secret(key)
print("secrets loaded:", ["HF_TOKEN"])

In [ ]:
# llm-compressor brings its own pins for transformers/compressed-tensors; this notebook does not
# run vLLM, so they cannot clash with the serving notebook's vllm==0.30.0.
!pip install -q llmcompressor peft
!pip uninstall -y -q torchaudio

# Step out of the repo BEFORE deleting it: rmtree on the kernel's own working
# directory leaves cwd pointing at a dead inode, and then every ! command fails
# with "getcwd: cannot access parent directories" -- including this git clone.
%cd /kaggle/working
import shutil
shutil.rmtree("/kaggle/working/the_llm_project", ignore_errors=True)
!git clone -q https://github.com/madhusiddharths/the_llm_project.git /kaggle/working/the_llm_project
%cd /kaggle/working/the_llm_project
!git log -1 --oneline

In [ ]:
# Finds the attached states dataset (make_pairs.py states output: states.jsonl + manifest.json).
import glob, json, os
hits = [h for h in glob.glob("/kaggle/input/**/manifest.json", recursive=True)
        if json.load(open(h)).get("file") == "states.jsonl"]
assert hits, "attach the tool-router-grpo-states dataset (Add Input): no states manifest under /kaggle/input"
STATES = os.path.join(os.path.dirname(hits[0]), "states.jsonl")
HF_USER = "madhusiddharths1"
print("STATES =", STATES)

In [ ]:
### Smoke: 0.5B, 8 samples of 512 tokens, not pushed (~5 min). Checks merge, GPTQ and save.
!python src/quantize.py --config configs/qwen05b.yaml --adapter "$HF_USER/tool-router-qwen05b-sft" --states "$STATES" --out /kaggle/working/smoke-gptq --smoke --samples 8 --max-len 512
assert _exit_code == 0, "quantize smoke failed: send the output above"
print(open("/kaggle/working/smoke-gptq/quantize.json").read())

In [ ]:
### The three sizes, smallest first; each is pushed as soon as it is done.
from huggingface_hub import HfApi
api = HfApi()
for size in ["qwen05b", "qwen15b", "qwen3b"]:
    repo = f"{HF_USER}/tool-router-{size}-sft-gptq"
    if api.repo_exists(repo) and api.file_exists(repo, "quantize.json"):
        print(f"===== {size}: already on the Hub, skipped =====")
        continue
    print(f"===== {size} =====")
    out = f"/kaggle/working/{size}-sft-gptq"
    !python src/quantize.py --config configs/{size}.yaml --adapter "$HF_USER/tool-router-{size}-sft" --states "$STATES" --out "{out}" --push "{repo}"
    assert _exit_code == 0, f"quantize failed on {size}: send the output above"
    print(open(f"{out}/quantize.json").read())